# HybriDock-Pep on Google Colab

Peptide–protein docking and calibrated ΔG scoring on a free Colab **T4 GPU**.
Give it a receptor PDB and a peptide sequence; get back ranked 3D poses and a
binding free energy in kcal/mol.

Repo: [Tasty-Ramen2010/hybridock-pep](https://github.com/Tasty-Ramen2010/hybridock-pep) ·
Creator: **Choppa Purandhar Ram**

---

### Before you start

**Set the runtime to a GPU:** *Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ **T4 GPU*** ▸ Save.
Do this first — installing on a CPU runtime installs the wrong PyTorch build.

### What this notebook does

| Step | Cell | Time (T4) |
|---|---|---|
| Check the GPU | 1 | seconds |
| Mount Drive (optional cache) | 2 | seconds |
| Clone the repo | 3 | ~30 s |
| Build both conda environments | 4 | **15–25 min**, once per session |
| Validate the install (`crystal-score`) | 5 | ~30 s |
| Dock your peptide | 6–7 | ~3–10 min at `--n-samples 100` |
| Inspect / view / download results | 8–11 | seconds |

Colab wipes the VM when the session ends, so step 4 runs again on every new
session. Mounting Drive in step 2 keeps the ~2.5 GB ESM-2 weights and the model
checkpoints between sessions, which saves several minutes each time — the conda
environments themselves are rebuilt regardless.

> **Free-tier note:** a Colab session is capped at roughly 12 h and can be
> reclaimed while idle. A `--n-samples 100` dock finishes well inside that, but
> start long `--ultra` runs early and keep the tab open.

## 1 · Check the GPU

In [ ]:
#@title Run me first — confirms a GPU is attached
import subprocess, sys

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    print("NO GPU DETECTED.\n"
          "Fix: Runtime > Change runtime type > Hardware accelerator > T4 GPU > Save,\n"
          "then re-run this cell. Everything below assumes a GPU is attached.")
else:
    print(out.stdout)
    name, cc = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True).stdout.strip().split(", ")
    print(f"GPU: {name}  (compute capability {cc})")
    print("T4 = compute capability 7.5 — supported; the setup script picks the "
          "matching CUDA build automatically.")

## 2 · Optional: mount Google Drive

Only worth doing if you plan to come back for a second session. It gives you:

* a persistent cache for the ESM-2 language-model weights (~2.5 GB) and the two
  RAPiDock checkpoints (~55 MB), so later sessions skip those downloads;
* a place to keep run outputs after the VM is recycled.

Set `USE_DRIVE = False` to skip it — everything still works, just session-local.

In [ ]:
#@title Mount Drive (optional)
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "MyDrive/hybridock-pep"  #@param {type:"string"}

import os
CACHE_DIR = ""
RESULTS_DIR = "/content/results"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    base = f"/content/drive/{DRIVE_FOLDER}"
    CACHE_DIR = f"{base}/cache"
    RESULTS_DIR = f"{base}/results"
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f"Cache:   {CACHE_DIR}")
else:
    print("Running without Drive — the cache and results live only in this session.")

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results: {RESULTS_DIR}")

## 3 · Clone the repository

In [ ]:
#@title Clone HybriDock-Pep
import os, subprocess

REPO_DIR = "/content/hybridock-pep"
BRANCH = "master"  #@param {type:"string"}

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--recursive", "--branch", BRANCH,
         "https://github.com/Tasty-Ramen2010/hybridock-pep.git", REPO_DIR],
        check=True)
else:
    # A reconnected (not deleted) runtime still has the previous clone. Reusing
    # it silently would run an older scripts/colab_setup.sh against this
    # notebook, so bring it up to date instead of assuming it is current.
    print("Repo already present — pulling latest.")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "submodule", "update",
                    "--init", "--recursive"], check=False)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
print(subprocess.run(["git", "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)

## 4 · Build the environments

This is the slow cell: **15–25 minutes**. It installs micromamba, builds
`score-env` (Vina, OpenMM, RDKit, meeko) and `rapidock` (the diffusion stack),
installs the PyTorch/PyG build matching *this* GPU, and downloads both RAPiDock
checkpoints from Zenodo.

Two envs, not one, because the sampling and scoring stacks have version
requirements that cannot be solved together — the scoring side drives the
sampling side as a subprocess.

Leave the tab open and let it run. Output is long; the last lines are a summary.

In [ ]:
#@title Install (15–25 min) — re-runnable, skips what is already built
LITE = False  #@param {type:"boolean"}

import subprocess, sys

cmd = ["bash", "scripts/colab_setup.sh"]
if CACHE_DIR:
    cmd += ["--cache-dir", CACHE_DIR]
if LITE:
    # Trims ~260 MB: the blind-mode checkpoint, the AD4/obabel extras, and the
    # Boost headers once Vina has compiled. Rules out `--blind` and `--scoring ad4`.
    # It cannot shrink PyTorch or the 2.4 GB ESM-2 weights -- docking needs both.
    cmd.append("--lite")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()

if proc.returncode != 0:
    raise SystemExit(f"setup failed (exit {proc.returncode}) — see the output above")

In [ ]:
#@title Wire up the paths the rest of the notebook uses
import os, subprocess, shlex, time

SCORE_PREFIX = "/opt/conda/envs/score-env"
RAPIDOCK_PREFIX = "/opt/conda/envs/rapidock"
HYBRIDOCK = f"{SCORE_PREFIX}/bin/hybridock-pep"

# The pipeline finds the sampling env on its own via /opt/conda, but being
# explicit costs nothing and makes a broken install fail loudly instead of
# silently dropping to CPU.
os.environ["RAPIDOCK_PYTHON"] = f"{RAPIDOCK_PREFIX}/bin/python3"
os.environ["RAPIDOCK_DIR"] = f"{REPO_DIR}/third_party/RAPiDock"

# Docking prints progress as it goes; capturing output and printing it at the
# end would leave you staring at a blank cell for ten minutes.
def run(args, env=None):
    "Run a command, streaming its output live, and return the exit code."
    print("$", " ".join(shlex.quote(a) for a in args), "\n", flush=True)
    started = time.time()
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, **(env or {})})
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    mins, secs = divmod(int(time.time() - started), 60)
    print(f"\n--- exit {proc.returncode} in {mins}m {secs}s ---")
    return proc.returncode

assert os.path.exists(HYBRIDOCK), "score-env is missing — re-run the install cell"
run([HYBRIDOCK, "--help"])

### Prefer a terminal to a notebook?

The setup above puts `hybridock-pep` and `hybridock-tui` on `PATH`, so you can
skip the rest of this notebook and drive the CLI directly — every command in the
README works verbatim.

If you have Colab Pro, use the terminal button at the bottom of the left sidebar.
On the free tier there is no terminal button, but the cell below gives you a real
one inside the notebook. Then:

```bash
cd /content/hybridock-pep
hybridock-tui                    # guided UI, nothing to memorize
# or drive it directly:
hybridock-pep dock --peptide ETFSDLWKLLPE \
    --receptor data/pdbs/1YCR_mdm2.pdb \
    --site 25.20 -25.61 -7.97 --box 30 \
    --n-samples 100 --output-dir runs/my_run
```

`hybridock-tui` is a full-screen application, and an in-notebook terminal is a
smaller, less capable terminal than a real one. If the layout renders badly or
the keys misbehave, `./launch_ui.sh --cli` is the same wizard as a plain
question-and-answer prompt, and `./launch_ui.sh --print` just builds the command
so you can copy it into a cell.

In [ ]:
#@title Optional — open a real terminal inside this notebook
!pip install -q colab-xterm
%load_ext colabxterm
%xterm

## 5 · Validate the install

`crystal-score` scores a known crystal complex — p53 peptide bound to MDM2
(PDB 1YCR). No sampling, no GPU, a few seconds.

**Expected: ΔG ≈ −9.3 kcal/mol** (experimental ≈ −8.5). Anything from −8 to −11
means the install is good; the value moves by up to ~1 kcal/mol with the
resolved scikit-learn/numpy versions, so a small difference is not a failure.

In [ ]:
#@title Sanity check — score a known crystal complex
run([HYBRIDOCK, "crystal-score",
     "--receptor", "data/pdbs/1YCR_mdm2.pdb",
     "--peptide-pdb", "data/pdbs/1YCR_peptide.pdb",
     "--peptide", "ETFSDLWKLLPE"])

## 6 · Choose what to dock

Three ways to supply a receptor:

* **example** — a bundled structure from `data/pdbs/` (MDM2 is the tutorial case).
* **pdb_id** — fetched straight from the RCSB (e.g. `1YCR`). Strip the ligands
  and waters yourself if the file has them; the pipeline docks against whatever
  protein you hand it.
* **upload** — pick a `.pdb` from your machine.

**About `--site` and `--box`:** `--site` is the pocket centre in Ångströms and
`--box` is the cube edge (30 Å suits 12-mers and longer). Don't know the pocket?
Tick `BLIND` and leave the site alone — the pipeline runs a real pocket search
(an exploratory whole-receptor pass, clustered into candidate sites, then
refinement at each). Blind mode costs roughly 3–5× a targeted run.

In [ ]:
#@title Docking inputs
PEPTIDE = "ETFSDLWKLLPE"  #@param {type:"string"}

RECEPTOR_SOURCE = "example"  #@param ["example", "pdb_id", "upload"]
EXAMPLE_RECEPTOR = "data/pdbs/1YCR_mdm2.pdb"  #@param ["data/pdbs/1YCR_mdm2.pdb", "data/pdbs/3LNJ_mdm2.pdb", "data/pdbs/hldh.pdb", "data/pdbs/1T2D_receptor.pdb"]
PDB_ID = "1YCR"  #@param {type:"string"}

BLIND = False  #@param {type:"boolean"}
SITE_X = 25.20  #@param {type:"number"}
SITE_Y = -25.61  #@param {type:"number"}
SITE_Z = -7.97  #@param {type:"number"}
BOX = 30  #@param {type:"number"}

N_SAMPLES = 100  #@param {type:"integer"}
REFINE_TOPK = 0  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}
RUN_NAME = "colab_run"  #@param {type:"string"}

import os, shutil, urllib.request

if RECEPTOR_SOURCE == "example":
    RECEPTOR = EXAMPLE_RECEPTOR
elif RECEPTOR_SOURCE == "pdb_id":
    RECEPTOR = f"/content/{PDB_ID.upper()}.pdb"
    urllib.request.urlretrieve(
        f"https://files.rcsb.org/download/{PDB_ID.upper()}.pdb", RECEPTOR)
    print(f"Fetched {PDB_ID.upper()} from the RCSB")
else:
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    RECEPTOR = f"/content/{name}"
    shutil.move(name, RECEPTOR)

OUTPUT_DIR = f"/content/hybridock-pep/runs/{RUN_NAME}"

assert os.path.exists(RECEPTOR), f"receptor not found: {RECEPTOR}"
print(f"Peptide  : {PEPTIDE}  ({len(PEPTIDE)} residues)")
print(f"Receptor : {RECEPTOR}")
print(f"Mode     : {'blind pocket search' if BLIND else f'targeted at ({SITE_X}, {SITE_Y}, {SITE_Z}), box {BOX} A'}")
print(f"Samples  : {N_SAMPLES}")
print(f"Output   : {OUTPUT_DIR}")

if len(PEPTIDE) >= 13:
    print("\nNote: peptides of 13+ residues route to a long-peptide checkpoint "
          "(longer_local.pt) that is not published on Zenodo. The run falls back "
          "to rapidock_local.pt and prints a warning — expected, not an error.")

## 7 · Dock

First run of a session downloads the ESM-2 language model (~2.5 GB) before
sampling starts — that download is cached, so later runs skip it.

Rough T4 timings for a 12-mer against a small receptor: ~2–4 min at
`--n-samples 100`, plus ~1 min of minimisation. `REFINE_TOPK` adds MM-GBSA on
the top clusters and costs several more minutes, growing roughly with the square
of receptor size.

In [ ]:
#@title Run the pipeline
cmd = [HYBRIDOCK, "dock",
       "--peptide", PEPTIDE,
       "--receptor", RECEPTOR,
       "--n-samples", str(N_SAMPLES),
       "--seed", str(SEED),
       "--output-dir", OUTPUT_DIR]

if BLIND:
    cmd.append("--blind")
else:
    cmd += ["--site", str(SITE_X), str(SITE_Y), str(SITE_Z), "--box", str(BOX)]

if REFINE_TOPK > 0:
    cmd += ["--refine-topk", str(REFINE_TOPK)]

rc = run(cmd)
if rc != 0:
    print("\nDocking failed. See the Troubleshooting section at the bottom.")

## 8 · Ranked poses

In [ ]:
#@title Top poses by ΔG
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/ranked_poses.csv")
cols = [c for c in ["rank", "pose_filename", "delta_g", "pooled_affinity_dg",
                    "mmgbsa_dg", "hybrid_score", "vina_score", "bsa",
                    "n_clash", "cluster_id"] if c in df.columns]

print(f"{len(df)} poses scored\n")
best = df.iloc[0]
print(f"Best pose: {best['pose_filename']}   ΔG = {best['delta_g']:.2f} kcal/mol\n")
df[cols].head(10)

In [ ]:
#@title Convergence and clustering plots
from IPython.display import Image, display
import os

for plot in ("convergence_plot.png", "silhouette_plot.png"):
    path = f"{OUTPUT_DIR}/{plot}"
    if os.path.exists(path):
        print(plot)
        display(Image(path))

## 9 · Look at the best pose

Receptor in grey cartoon, the docked peptide in coloured sticks. Drag to rotate,
scroll to zoom.

In [ ]:
#@title 3D view of best_pose.pdb
!pip install -q py3Dmol
import py3Dmol

view = py3Dmol.view(width=900, height=600)
view.addModel(open(RECEPTOR).read(), "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "lightgrey"}})
view.addModel(open(f"{OUTPUT_DIR}/best_pose.pdb").read(), "pdb")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "cyanCarbon", "radius": 0.25},
                             "cartoon": {"color": "cyan"}})
view.zoomTo({"model": 1})
view.show()

## 10 · Save your results

In [ ]:
#@title Copy to Drive and/or download a zip
import shutil, os

archive = shutil.make_archive(f"/content/{RUN_NAME}", "zip", OUTPUT_DIR)
print(f"Archive: {archive} ({os.path.getsize(archive) / 1e6:.1f} MB)")

if USE_DRIVE:
    dest = f"{RESULTS_DIR}/{RUN_NAME}"
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(OUTPUT_DIR, dest)
    print(f"Copied to Drive: {dest}")

DOWNLOAD_ZIP = True  #@param {type:"boolean"}
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(archive)

## 11 · Other things this can do

Everything below runs in the same session — the environments are already built.
Full flag reference: `hybridock-pep guide all`.

### Selectivity — does the peptide prefer target A over target B?

Docks the same peptide against both receptors and reports ΔΔG with a bootstrap
95% CI. Each side needs its own site and box — they are different structures in
different coordinate frames.

```python
run([HYBRIDOCK, "selectivity",
     "--peptide", PEPTIDE,
     "--target-receptor", "data/pdbs/1YCR_mdm2.pdb",
     "--target-site", "25.20", "-25.61", "-7.97", "--target-box", "30",
     "--offtarget-receptor", "data/pdbs/3LNJ_mdm2.pdb",
     "--offtarget-site", "5.99", "5.87", "20.51", "--offtarget-box", "30",
     "--n-samples", "100",
     "--output-dir", "/content/hybridock-pep/runs/selectivity"])
```

The off-target coordinates above are a placeholder — find the real pocket centre
in your own structure first. `run([HYBRIDOCK, "guide", "selectivity"])` prints
the full flag reference.

### Blind docking — no known pocket

Tick `BLIND` in step 6 and leave the site fields alone. Costs ~3–5× a targeted
run: an exploratory whole-receptor pass, spatial clustering into candidate
pockets, then refinement at each.

### Ultra accuracy mode

`--ultra` adds randomised-smoothing ranking, MM-GBSA, interaction entropy, and
a charged-residue correction. This is the verification tier and is expensive —
budget an hour or more on a T4, and be aware Colab may reclaim an idle session.

---

## Troubleshooting

| Symptom | What it means | Fix |
|---|---|---|
| `NO GPU DETECTED` | CPU runtime | Runtime ▸ Change runtime type ▸ T4 GPU, then re-run from step 1 |
| Setup cell fails partway | Usually a flaky download | Re-run it — it skips what already built |
| `no kernel image is available` | Wrong CUDA build for this GPU | Re-run setup with `--force`; it verifies with a real kernel launch and falls back automatically |
| Stage 1 dies in ~10 s with `undefined symbol: ...parseSchemaOrName...` | torch and the PyG extensions were built against different C++ ABIs — conda-forge's torch got left in place | Re-run the setup cell (the current script pins `torch==...+cu124`, which forces the replacement) |
| Stage 1 crawls, no GPU use | The sampling env fell back to CPU | `!/opt/conda/envs/rapidock/bin/python3 -c "import torch; print(torch.cuda.is_available())"` — if `False`, re-run setup with `--force` |
| `longer_local.pt` warning | Peptide ≥13 residues | Expected. That checkpoint is not published; the run falls back to `rapidock_local.pt` |
| `receptor prep fell back to obabel` | meeko cannot import RDKit | Re-run setup with `--force` |
| Session died mid-run | Colab reclaimed an idle VM | Keep the tab active; for long runs mount Drive so finished output survives |
| Out of disk | Both envs plus weights are ~15 GB | Runtime ▸ Disconnect and delete runtime, then start fresh |
| ESM-2 download stalls or errors with Drive mounted | Drive's FUSE layer is unreliable for a 2.5 GB write | Set `USE_DRIVE = False` in step 2 and re-run setup — it downloads to local disk instead |

Two identical GPU runs differ by ~2.9 Å mean pose RMSD even with `--seed`.
That is CUDA nondeterminism in sampling, not a bug.

---

## Citing

If this contributes to published work, cite the Zenodo record:
[10.5281/zenodo.21680573](https://doi.org/10.5281/zenodo.21680573).
See the repository README for the full citation list, including RAPiDock,
AutoDock Vina, OpenMM, and ESM-2.